# Capítulo 9 - Exercícios Resolvidos

**Quantum Computing for the Quantum Curious - Hughes et al.**

Seção 9.7 - Check Your Understanding

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import numpy as np

## Exercício 1

**(a) Quantas peças de informação clássica podem ser representadas por 8 bits clássicos (1 byte)?**

In [ ]:
# Resposta (a): 2^8 = 256 peças de informação clássica
n_bits = 8
classical_states = 2**n_bits
print(f"8 bits clássicos podem representar {classical_states} estados diferentes")
print(f"Cada bit pode ser 0 ou 1: 2×2×2×2×2×2×2×2 = {2**8}")

**(b) E um computador quântico com 8 qubits?**

In [ ]:
# Resposta (b): Também 256 peças de informação clássica
# Quando medimos 8 qubits, obtemos um dos 256 estados clássicos possíveis
n_qubits = 8
quantum_states = 2**n_qubits
print(f"8 qubits, quando MEDIDOS, produzem um de {quantum_states} resultados")
print(f"Estados: |00000000⟩ até |11111111⟩")

**(c) Qual a vantagem do computador quântico sobre o clássico?**

In [ ]:
# Resposta (c): SUPERPOSIÇÃO permite processar todas as possibilidades simultaneamente
print("""
VANTAGEM QUÂNTICA:

1. SUPERPOSIÇÃO:
   - O computador quântico pode criar uma superposição de até 256 
     possibilidades simultaneamente
   - Realiza computação em TODAS elas ao mesmo tempo (paralelismo quântico)

2. PORÉM:
   - Quando fazemos uma medição, obtemos apenas UM único valor clássico
   - A medição "colapsa" a superposição

3. A ARTE da computação quântica:
   - Usar INTERFERÊNCIA para amplificar as respostas corretas
   - E cancelar as respostas incorretas ANTES da medição
""")

## Exercício 2

**Refere-se ao setup experimental da Figura 9.3 (interferômetro de Mach-Zehnder).**

Quais detectores disparam para cada função?

In [ ]:
# Interferômetro de Mach-Zehnder implementa o algoritmo de Deutsch
print("""
FUNÇÕES E DETECTORES:

┌──────────────────┬─────────────────┬───────────────┐
│ Função           │ Tipo            │ Detector      │
├──────────────────┼─────────────────┼───────────────┤
│ f₁(x) = 0        │ CONSTANTE       │ Detector 1    │
│ f₂(x) = 1        │ CONSTANTE       │ Detector 1    │
│ f₃(x) = x        │ BALANCEADA      │ Detector 2    │
│ f₄(x) = NOT(x)   │ BALANCEADA      │ Detector 2    │
└──────────────────┴─────────────────┴───────────────┘

A interferência construtiva/destrutiva depende se a função é 
constante ou balanceada, NÃO do valor específico.
""")

## Exercício 3

**(a) Quais detectores disparam se a função é constante?**

**(b) Quais detectores disparam se a função é balanceada?**

**(c) Quantos fótons você precisaria enviar para determinar o tipo da função?**

In [ ]:
print("""
RESPOSTAS:

(a) Função CONSTANTE → Apenas Detector 1 dispara

(b) Função BALANCEADA → Apenas Detector 2 dispara

(c) Apenas 1 FÓTON é necessário!

    VANTAGEM QUÂNTICA:
    - Classicamente: precisaríamos avaliar a função pelo menos 2 vezes
    - Quanticamente: 1 consulta é suficiente!
""")

## Exercício 4

**Explique como superposição e interferência permitem que o algoritmo Deutsch-Jozsa supere o algoritmo clássico.**

In [ ]:
print("""
SUPERPOSIÇÃO E INTERFERÊNCIA NO DEUTSCH-JOZSA:

1. SUPERPOSIÇÃO:
   - Aplicamos Hadamard em todos os qubits de entrada
   - Isso cria superposição de TODOS os valores de x
   - Avaliamos f(x) para todos os x SIMULTANEAMENTE

2. INTERFERÊNCIA:
   - Após aplicar o oráculo, aplicamos Hadamard novamente
   - As amplitudes de probabilidade INTERFEREM:
     
     Se f é CONSTANTE:
       → Interferência CONSTRUTIVA no estado |00...0⟩
     
     Se f é BALANCEADA:
       → Interferência DESTRUTIVA no estado |00...0⟩

3. RESULTADO:
   - Uma ÚNICA medição revela o tipo da função:
     • |00...0⟩ → função CONSTANTE
     • qualquer outro → função BALANCEADA

COMPARAÇÃO:
┌─────────────┬────────────────────────────────┐
│ Clássico    │ Até 2^(n-1) + 1 avaliações     │
│ Quântico    │ SEMPRE 1 avaliação             │
└─────────────┴────────────────────────────────┘
""")

In [ ]:
# Demonstração visual do Deutsch-Jozsa
def create_dj_circuit(oracle_type: str, n_qubits: int = 3):
    """Cria circuito Deutsch-Jozsa com oráculo especificado."""
    qc = QuantumCircuit(n_qubits + 1, n_qubits)
    
    # Inicialização
    qc.x(n_qubits)  # Auxiliar em |1⟩
    qc.h(range(n_qubits + 1))  # Hadamard em todos
    qc.barrier()
    
    # Oráculo
    if oracle_type == "constant_0":
        pass  # f(x) = 0, não faz nada
    elif oracle_type == "constant_1":
        qc.x(n_qubits)  # f(x) = 1
    elif oracle_type == "balanced":
        for i in range(n_qubits):
            qc.cx(i, n_qubits)  # f(x) = paridade
    
    qc.barrier()
    
    # Hadamard final e medição
    qc.h(range(n_qubits))
    qc.measure(range(n_qubits), range(n_qubits))
    
    return qc

# Testar com função constante
print("Circuito Deutsch-Jozsa (função CONSTANTE):")
qc_const = create_dj_circuit("constant_0", 3)
print(qc_const.draw())

simulator = AerSimulator()
result = simulator.run(qc_const, shots=1000).result()
counts = result.get_counts()
print(f"\nResultado: {counts}")
print("→ Apenas |000⟩ aparece → CONSTANTE")

In [ ]:
# Testar com função balanceada
print("Circuito Deutsch-Jozsa (função BALANCEADA):")
qc_bal = create_dj_circuit("balanced", 3)
print(qc_bal.draw())

result = simulator.run(qc_bal, shots=1000).result()
counts = result.get_counts()
print(f"\nResultado: {counts}")
print("→ Resultados diferentes de |000⟩ → BALANCEADA")

## Exercício 5

**A Figura 9.7 mostra a implementação de portas para testar uma função de 3 qubits.**

**(a) Quantas avaliações seriam necessárias classicamente?**

**(b) Executando no IBM Q, você consegue determinar o tipo da função?**

In [ ]:
print("""
(a) COMPLEXIDADE CLÁSSICA:

Para 3 qubits: 2³ = 8 valores possíveis

Pior caso: precisa verificar pelo menos metade + 1 = 5 avaliações

Por quê? Se os primeiros 4 resultados são iguais, ainda pode ser 
balanceada (os outros 4 podem ser diferentes).

============================================================

(b) RESULTADO NO IBM Q:

O histograma típico mostra:
  |001⟩: ~26.5%
  |011⟩: ~24.5%
  |101⟩: ~24.8%
  |111⟩: ~24.2%

Como existem resultados DIFERENTES de |000⟩:
→ A função é BALANCEADA!

Se fosse constante, veríamos apenas |000⟩ com ~100%.
""")

In [ ]:
# Simulação do resultado do IBM Q
# (simulando ruído típico de hardware real)
np.random.seed(42)

qc_ibm_sim = create_dj_circuit("balanced", 3)
result = simulator.run(qc_ibm_sim, shots=1000).result()
counts = result.get_counts()

print("Simulação (aproximando resultado do IBM Q):")
for state, count in sorted(counts.items()):
    print(f"  |{state}⟩: {count/10:.1f}%")

# Verificar se é balanceada
if '000' not in counts or counts.get('000', 0) < 100:
    print("\n→ Função é BALANCEADA (resultado ≠ |000⟩)")

## Resumo do Capítulo 9

### Conceitos-Chave:

1. **Paralelismo quântico** via superposição
2. **Algoritmo Deutsch-Jozsa**: 1 consulta vs. 2^(n-1)+1 clássico
3. **Interferência** amplifica respostas corretas e cancela incorretas